In [ ]:
SHOW TABLES IN multisport_training.training;

In [ ]:
SELECT *
FROM multisport_training.training.samples_cycling
ORDER BY timestamp DESC
LIMIT 10;

In [ ]:
SELECT *
FROM multisport_training.training.samples_running
ORDER BY exercise_index DESC
LIMIT 10;

In [ ]:
SELECT
  session_id,
  MAX(heart_rate) AS max_heart_rate,
  MIN(timestamp) AS session_start_time
FROM multisport_training.training.samples_cycling
GROUP BY session_id
ORDER BY session_start_time ASC

In [ ]:
WITH max_hr_session AS (
  SELECT session_id
  FROM multisport_training.training.samples_cycling
  ORDER BY heart_rate DESC
  LIMIT 1
)
SELECT
  FLOOR(second / 60) AS minute,
  AVG(heart_rate) AS avg_heart_rate
FROM multisport_training.training.samples_cycling
WHERE session_id IN (SELECT session_id FROM max_hr_session)
GROUP BY FLOOR(second / 60)
ORDER BY minute ASC

In [ ]:
SELECT
  session_id,
  MAX(heart_rate) AS max_heart_rate,
  MIN(timestamp) AS session_start_time
FROM multisport_training.training.samples_running
GROUP BY session_id
ORDER BY session_start_time ASC

In [ ]:
SELECT *
FROM multisport_training.training.samples_running
ORDER BY exercise_index DESC
LIMIT 10;

In [ ]:
SELECT
  session_id,
  second,
  AVG(heart_rate) AS heart_rate_avg,
  (MAX(distance_m) - MIN(distance_m)) / 5 * 3.6 AS calculated_speed_kmh,
  AVG(speed_kmh) AS avg_speed_kmh,
  (MAX(distance_m) - MIN(distance_m)) AS distance_for_increment_m,
  try_divide(AVG(heart_rate), (MAX(distance_m) - MIN(distance_m)) / 5 * 3.6) AS hr_per_speed_kmh
FROM multisport_training.training.samples_running
WHERE session_id = '87c7e57e796cda60'
GROUP BY session_id, second
ORDER BY session_id ASC, second ASC

In [ ]:
SELECT
  session_id,
  MIN(timestamp) AS session_start_time,
  AVG(heart_rate) / AVG(speed_kmh) AS avg_hr_per_speed_kmh,
  AVG(temperature_c) AS avg_temperature_c,
  FLOOR(AVG(temperature_c) / 10) * 10 AS avg_temperature_group
FROM (
  SELECT *,
    distance_m - LAG(distance_m) OVER (PARTITION BY session_id ORDER BY second) AS distance_diff
  FROM multisport_training.training.samples_running
)
WHERE (distance_diff IS NULL OR distance_diff <> 0)
  AND speed_kmh <= 28
GROUP BY session_id
ORDER BY session_start_time ASC

In [ ]:
SELECT *
FROM multisport_training.training.sessions
LIMIT 10;